In [ ]:
import sys
import warnings
import time
from pathlib import Path

warnings.filterwarnings("ignore")

try:
    sys.path.insert(0, str(Path('__file__').resolve().parents[1]))
except:
    sys.path.insert(0, str(Path.cwd().parent))

import shared
import numpy as np
import geopandas as gpd
import networkx as nx
import rasterio
import matplotlib.pyplot as plt
from rasterio.features import rasterize as rio_rasterize
from rasterio.transform import rowcol
from rasterio.enums import MergeAlg
from shapely.geometry import Point, LineString
from scipy.spatial import distance_matrix

# =====================================================================
# 1. VARIABLES AND PATHS
# =====================================================================
cost_path = shared.OUT / "cost_surface_tobler.tif"
figures_dir = shared.OUT / "figures" / "proximity"
figures_dir.mkdir(parents=True, exist_ok=True)
vector_out_dir = shared.OUT / "vector_gis"
vector_out_dir.mkdir(parents=True, exist_ok=True)

edge_density_path = shared.OUT / "proximity_edge_density.tif"
steiner_tree_path = vector_out_dir / "proximity_steiner_tree.geojson"

# Load base data
footprints, doors_gdf, doors_pts, crosswalk = shared.get_base_preprocessed_data()
points = np.array([[geom.x, geom.y] for geom in doors_pts.geometry])

In [ ]:
# =====================================================================
# 2. HELPER FUNCTIONS
# =====================================================================
def create_gabriel_graph(points, crs):
    num_points = len(points)
    graph = nx.Graph()
    graph.graph['crs'] = crs
    
    for i in range(num_points):
        graph.add_node(i, x=float(points[i, 0]), y=float(points[i, 1]))
        
    if num_points < 2:
        return graph
        
    dist_matrix = distance_matrix(points, points)
    for i in range(num_points):
        for j in range(i + 1, num_points):
            midpoint = (points[i] + points[j]) / 2
            radius = dist_matrix[i, j] / 2
            other_points = np.delete(points, [i, j], 0)
            
            if len(other_points) == 0 or not np.any(np.linalg.norm(other_points - midpoint, axis=1) < radius - 1e-9):
                graph.add_edge(i, j, weight=float(dist_matrix[i, j]))
                
    return graph

def create_beta_skeleton(points, beta=1.5):
    num_points = len(points)
    graph = nx.Graph()
    
    for i in range(num_points):
        graph.add_node(i, x=float(points[i, 0]), y=float(points[i, 1]))
        
    if num_points < 2:
        return graph
        
    dist_matrix = distance_matrix(points, points)
    for i in range(num_points):
        for j in range(i + 1, num_points):
            dist = dist_matrix[i, j]
            radius = beta / 2 * dist
            center1 = (1 - beta / 2) * points[i] + beta / 2 * points[j]
            center2 = beta / 2 * points[i] + (1 - beta / 2) * points[j]
            other_points = np.delete(points, [i, j], 0)
            
            if len(other_points) == 0 or not np.any((np.linalg.norm(other_points - center1, axis=1) < radius - 1e-9) & (np.linalg.norm(other_points - center2, axis=1) < radius - 1e-9)):
                graph.add_edge(i, j, weight=float(dist))
                
    return graph

def filter_barrier_edges(graph, cost_surface, transform, points, pct=95):
    valid_costs = cost_surface[np.isfinite(cost_surface)]
    if not len(valid_costs):
        return graph
        
    threshold = np.percentile(valid_costs, pct)
    height, width = cost_surface.shape
    edges_to_remove = []
    
    for u, v in graph.edges():
        point1, point2 = points[u], points[v]
        num_samples = max(10, int(np.ceil(np.linalg.norm(point2 - point1) / 0.2)))
        t_vals = np.linspace(0, 1, num_samples)
        sample_points = point1 + t_vals[:, None] * (point2 - point1)
        rows, cols = rowcol(transform, sample_points[:, 0], sample_points[:, 1])
        
        is_bad_edge = any(
            (0 <= r < height and 0 <= c < width and (np.isnan(cost_surface[r, c]) or cost_surface[r, c] > threshold)) or 
            not (0 <= r < height and 0 <= c < width) 
            for r, c in zip(rows, cols)
        )
        
        if is_bad_edge:
            edges_to_remove.append((u, v))
            
    graph.remove_edges_from(edges_to_remove)
    return graph

def compute_steiner_tree(graph, terminals, points):
    from networkx.algorithms.approximation import steiner_tree as nx_st
    
    for i in graph.nodes():
        graph.nodes[i]['x'] = float(points[i, 0])
        graph.nodes[i]['y'] = float(points[i, 1])
        
    if graph.number_of_edges() == 0:
        return graph
        
    if not nx.is_connected(graph):
        components = sorted(nx.connected_components(graph), key=len, reverse=True)
        best_component = max(components, key=lambda c: len(c & set(terminals)))
        graph = graph.subgraph(best_component).copy()
        terminals = [t for t in terminals if t in best_component]
        
    if len(terminals) <= 1:
        return nx.Graph()
        
    try:
        tree = nx_st(graph, terminals, weight='weight')
    except:
        tree = nx.minimum_spanning_tree(graph, weight='weight')
        
    for n in tree.nodes():
        tree.nodes[n]['x'] = float(points[n, 0])
        tree.nodes[n]['y'] = float(points[n, 1])
        
    return tree

def generate_edge_density_raster(gabriel_graph, beta_graphs, points, shape, transform):
    graphs = [gabriel_graph] + list(beta_graphs)
    geometries = [(LineString([points[u], points[v]]), 1.0) for g in graphs for u, v in g.edges()]
    
    if not geometries:
        return np.zeros(shape, dtype=np.float32)
        
    return rio_rasterize(geometries, out_shape=shape, transform=transform, fill=0, merge_alg=MergeAlg.add, dtype=np.float32)

In [ ]:
# =====================================================================
# 3. COST SURFACE PREPARATION
# =====================================================================
if not cost_path.exists():
    dem_data = shared.load_dem()
    footprints_dem_crs = footprints.to_crs(dem_data["crs"]) if footprints.crs != dem_data["crs"] else footprints.copy()
    
    dy, dx = np.gradient(dem_data["disp"], dem_data["res"], dem_data["res"])
    cost_tobler = 1.0 / np.maximum(6.0 * np.exp(-3.5 * np.abs(np.sqrt(dx**2 + dy**2) + 0.05)) / 3.6, 1e-6)
    
    build_mask = rio_rasterize(
        [(geom, 1) for geom in footprints_dem_crs.geometry if geom], 
        out_shape=dem_data["arr"].shape, 
        transform=dem_data["transform"], 
        fill=0, 
        dtype=np.uint8
    )
    
    cost_surface = 0.5 * cost_tobler + np.where(build_mask == 1, 1e9, 0.0)
    cost_profile = dem_data["profile"].copy()
    cost_profile.update(dtype="float32", count=1)
    
    with rasterio.open(cost_path, "w", **cost_profile) as dest:
        dest.write(cost_surface.astype(np.float32), 1)
        
    cost_transform = dem_data["transform"]
    cost_shape = cost_surface.shape
else:
    with rasterio.open(str(cost_path)) as source:
        cost_surface = source.read(1)
        cost_transform = source.transform
        cost_shape = cost_surface.shape
        cost_profile = source.profile

In [ ]:
# =====================================================================
# 4. COMPUTE GRAPHS
# =====================================================================
gabriel = create_gabriel_graph(points, footprints.crs)
beta_skel = create_beta_skeleton(points, beta=1.5)

gabriel_clean = filter_barrier_edges(gabriel, cost_surface, cost_transform, points)
beta_clean = filter_barrier_edges(beta_skel, cost_surface, cost_transform, points)

steiner_tree = compute_steiner_tree(gabriel_clean, list(range(len(points))), points)

In [ ]:
# =====================================================================
# 5. GRAPH/PLOTTING
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for ax, graph, title in [(axes[0], gabriel_clean, "Gabriel Graph (filtered)"), (axes[1], steiner_tree, "Steiner Tree")]:
    footprints.plot(ax=ax, color="#eef3f7", edgecolor="#b0c4de", linewidth=0.5, alpha=0.9)
    for u, v in graph.edges():
        ax.plot([graph.nodes[u]["x"], graph.nodes[v]["x"]], 
                [graph.nodes[u]["y"], graph.nodes[v]["y"]],
                color="#2980b9", linewidth=0.8, alpha=0.7)
    ax.scatter(points[:, 0], points[:, 1], s=10, c="red", zorder=5, label="Doors")
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.legend(fontsize=9)
    
plt.suptitle("Proximity Graphs — Gabriel Graph & Steiner Tree", fontsize=16, fontweight="bold")
plt.tight_layout()
shared.save_fig(figures_dir, "19_proximity_graphs.png", dpi=150)
plt.show()

In [ ]:
# =====================================================================
# 6. DATA EXPORT
# =====================================================================
edge_density = generate_edge_density_raster(gabriel_clean, [beta_clean], points, cost_shape, cost_transform)
density_profile = cost_profile.copy()
density_profile.update(dtype="float32", count=1, nodata=0.0)

with rasterio.open(str(edge_density_path), "w", **density_profile) as dest:
    dest.write(edge_density.astype(np.float32), 1)

tree_edges = [{"geometry": LineString([(steiner_tree.nodes[u]['x'], steiner_tree.nodes[u]['y']), 
                                       (steiner_tree.nodes[v]['x'], steiner_tree.nodes[v]['y'])]),
               "node_u": int(u), "node_v": int(v), "weight": float(d.get("weight", 0))} 
              for u, v, d in steiner_tree.edges(data=True)]

if tree_edges:
    edges_gdf = gpd.GeoDataFrame(tree_edges, crs=footprints.crs)
    edges_gdf.to_crs("EPSG:4326").to_file(str(steiner_tree_path), driver="GeoJSON")